In [ ]:
import openpyxl
import math


In [ ]:
path = "Dataset_TD4.xlsx"
wb = openpyxl.load_workbook(path, data_only=True)
ws = wb.active
headers = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
idx = {h: i for i, h in enumerate(headers)}
rows = []
for r in ws.iter_rows(min_row=2, values_only=True):
    if r is None:
        continue
    rows.append(r)
rows = [r for r in rows if r[idx["transaction date (1=1day=24 hours)"]] is not None]


In [ ]:
rows.sort(key=lambda r: r[idx["transaction date (1=1day=24 hours)"]])
dates = [r[idx["transaction date (1=1day=24 hours)"]] for r in rows]
spread = [float(r[idx["bid-ask spread"]]) if r[idx["bid-ask spread"]] is not None else float("nan") for r in rows]
sign = [float(r[idx["Sign of the transaction"]]) if r[idx["Sign of the transaction"]] is not None else float("nan") for r in rows]
price = [float(r[idx["Price (before transaction)"]]) if r[idx["Price (before transaction)"]] is not None else float("nan") for r in rows]
n0 = len(rows)
n0


In [ ]:
valid = []
for i in range(n0):
    if not (math.isnan(spread[i]) or math.isnan(sign[i]) or math.isnan(price[i])):
        valid.append(i)
rows2 = [rows[i] for i in valid]
dates = [dates[i] for i in valid]
spread = [spread[i] for i in valid]
sign = [sign[i] for i in valid]
price = [price[i] for i in valid]
n = len(price)
n


In [ ]:
dp = []
eps_t = []
eps_tm1 = []
spr_t = []
for i in range(1, n - 1):
    dp.append(price[i + 1] - price[i])
    eps_t.append(sign[i])
    eps_tm1.append(sign[i - 1])
    spr_t.append(spread[i])
N = len(dp)
N


In [ ]:
def transpose(A):
    return [list(row) for row in zip(*A)]

def matmul(A, B):
    m = len(A)
    n = len(A[0]) if m else 0
    p = len(B[0]) if len(B) else 0
    out = [[0.0] * p for _ in range(m)]
    for i in range(m):
        Ai = A[i]
        for k in range(n):
            aik = Ai[k]
            Bk = B[k]
            for j in range(p):
                out[i][j] += aik * Bk[j]
    return out

def eye(n):
    I = [[0.0] * n for _ in range(n)]
    for i in range(n):
        I[i][i] = 1.0
    return I

def inv(A):
    n = len(A)
    M = [row[:] + Irow[:] for row, Irow in zip(A, eye(n))]
    for col in range(n):
        pivot = col
        maxv = abs(M[col][col])
        for r in range(col + 1, n):
            v = abs(M[r][col])
            if v > maxv:
                maxv = v
                pivot = r
        if maxv == 0.0:
            raise ValueError("Singular matrix")
        if pivot != col:
            M[col], M[pivot] = M[pivot], M[col]
        piv = M[col][col]
        invp = 1.0 / piv
        for j in range(2 * n):
            M[col][j] *= invp
        for r in range(n):
            if r == col:
                continue
            factor = M[r][col]
            if factor != 0.0:
                for j in range(2 * n):
                    M[r][j] -= factor * M[col][j]
    return [row[n:] for row in M]

def vec_to_col(v):
    return [[float(x)] for x in v]

def mean(v):
    return sum(v) / len(v)

def diag(A):
    return [A[i][i] for i in range(len(A))]


In [ ]:
def ols(X, y):
    Xt = transpose(X)
    XtX = matmul(Xt, X)
    XtX_inv = inv(XtX)
    Xty = matmul(Xt, y)
    beta = matmul(XtX_inv, Xty)
    y_hat = matmul(X, beta)
    resid = [[y[i][0] - y_hat[i][0]] for i in range(len(y))]
    y_mean = mean([yy[0] for yy in y])
    ss_tot = sum((yy[0] - y_mean) ** 2 for yy in y)
    ss_res = sum(rr[0] ** 2 for rr in resid)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0.0 else float("nan")
    n = len(y)
    k = len(X[0])
    sigma2 = ss_res / (n - k)
    cov_beta = [[sigma2 * XtX_inv[i][j] for j in range(k)] for i in range(k)]
    se = [math.sqrt(cov_beta[i][i]) for i in range(k)]
    tstats = [beta[i][0] / se[i] if se[i] != 0.0 else float("nan") for i in range(k)]
    return beta, se, tstats, r2, n


In [ ]:
y = vec_to_col(dp)
X_full = []
for i in range(N):
    X_full.append([1.0, eps_t[i], eps_tm1[i], spr_t[i]])
beta_f, se_f, t_f, r2_f, n_f = ols(X_full, y)
names_f = ["const", "lambda (eps_t)", "eta (eps_tm1)", "beta (spread)"]
print(f"[FULL] N={n_f}, k={len(names_f)}, R2={r2_f:.6f}\n")
for nm, b, s, t in zip(names_f, [bb[0] for bb in beta_f], se_f, t_f):
    print(f"{nm:16s} coef={b: .6e}   se={s:.2e}   t={t:.2f}")


In [ ]:
X_core = []
for i in range(N):
    X_core.append([1.0, eps_t[i], eps_tm1[i]])
beta_c, se_c, t_c, r2_c, n_c = ols(X_core, y)
names_c = ["const", "lambda (eps_t)", "eta (eps_tm1)"]
print(f"\n[CORE] N={n_c}, k={len(names_c)}, R2={r2_c:.6f}\n")
for nm, b, s, t in zip(names_c, [bb[0] for bb in beta_c], se_c, t_c):
    print(f"{nm:16s} coef={b: .6e}   se={s:.2e}   t={t:.2f}")
